# CEP: Efficient Market Hypothesis

In [1]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('../../../')

In [2]:
import pandas as pd
from statsmodels.formula import api as smf
from data_helpers.data_loaders import PandasDataLoader 
from data_helpers.data_prep import dynamic_treatments
import numpy as np
from tqdm.auto import tqdm


## Load and filter dataset

In [3]:
model_name = 'emh'
sample_splits_df = pd.read_feather('../../../data/preprocessed/train_test_split.ft')
time_aggregated_dataset = pd.read_feather('../../../data/preprocessed/time_aggregate_dataset.ft')
time_aggregated_dataset = time_aggregated_dataset[~time_aggregated_dataset.treatment.isin(dynamic_treatments)]
time_aggregated_dataset = time_aggregated_dataset.query('round <= 5 and time <= 120')

## Keep relevant columns and prepare train-test loader

In [4]:
key_columns = ['treatment', 'game', 'round', 'time', 'n_unique_deals_round']
rounds = range(1,5)
n_deal_prices = range(0,6)
pdl = PandasDataLoader(sample_splits_df, time_aggregated_dataset)
target_col = 'ce_round'

## Fit and evaluate models

In [5]:
np.random.seed(1)
all_results = []
for i in tqdm(range(pdl.max_samples)):
    # Load the test set. As deal prices are used from the observed data, this method does not use the train set.
    _, test_df = pdl.get_sample_split_dataset(i)
    for rd in rounds:
        for n_deal_price in n_deal_prices:
            test_query = 'round == ' + str(rd) + ' and n_unique_deals_round ==  ' + str(n_deal_price)
            sub_test_set = test_df.query(test_query)
            prediction = sub_test_set['realized_price']
            test_targets = sub_test_set[target_col]

            # Persist performance results in terms of APE.
            result_test_df = sub_test_set[key_columns].copy()
            result_test_df.loc[:, 'ce_ape'] = (np.abs(prediction - test_targets)/test_targets)
            result_test_df.loc[:, 'sample_id'] = i
            all_results.append(result_test_df)

all_results_df = pd.concat(all_results, ignore_index = True)
all_results_df['model'] = model_name

  0%|          | 0/50 [00:00<?, ?it/s]

## Save results to file

In [7]:
all_results_df.reset_index().to_feather('../../../data/results/ce_price/'+model_name+'.ft')